# 🎓 Proyecto: Detección de Falsificación de Comprobantes de Pago Digital (Nequi)
## Asignatura: Inteligencia Artificial Avanzada | Metodología: Aprendizaje Basado en Retos (ABR)
---
### 📌 1. Identificación y Justificación del Problema
En Colombia y Latinoamérica, el auge de las billeteras digitales (como Nequi) ha transformado las finanzas cotidianas. Nequi actualizó recientemente su formato oficial a un **diseño de tiquete con código QR de verificación dinámica, fondo claro con ilustraciones y estructura estandarizada**.

A pesar de esta actualización, los ciberdelincuentes siguen utilizando:
1. **Edición digital en Photoshop/Canva:** Alterando el monto (`¿Cuánto?`), fecha o teléfono sobre capturas legítimas.
2. **Aplicaciones clonadas fraudulentas ("Nequi Fake"):** Imitando la pantalla con códigos QR falsos, colores menta descalibrados y fuentes del sistema.
3. **Reutilización de comprobantes antiguos:** Modificando la fecha y la referencia para engañar a comercios.

**Objetivo del Modelo de IA:** Implementar un sistema de Inteligencia Artificial que combine **Análisis Forense Digital (Error Level Analysis - ELA)** y **Redes Neuronales Convolucionales (CNN / MobileNetV3)** para evaluar comprobantes y determinar con precisión si son auténticos o falsificados.

In [ ]:
# ====================================================================
# 0. INSTALACIÓN Y CONFIGURACIÓN DEL ENTORNO
# ====================================================================
import os
import random
import glob
from io import BytesIO
from datetime import datetime, timedelta

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageChops, ImageEnhance
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Fijar semillas para reproducibilidad científica
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Entorno configurado correctamente. Dispositivo de aceleración: {device}")

--- 
### 🧪 2. Generador del Nuevo Formato Oficial de Nequi (QR + Tiquete)
Implementamos el renderizado programático del **diseño actual de Nequi** (código QR con marco verde menta `#84E4BD`, tipografía oficial, etiquetas `Para`, `¿Cuánto?`, `Número Nequi`, `Fecha` y `Referencia` M-XXXXXXXX).

In [ ]:
# Colores oficiales del nuevo diseño Nequi
COLOR_MINT_QR = (132, 228, 189)     # Verde menta/cian (#84E4BD)
COLOR_PURPLE = (32, 4, 34)          # Morado Nequi (#200422)
COLOR_TEXT_DARK = (20, 20, 25)      # Texto principal
COLOR_LABEL_GRAY = (110, 110, 120)  # Etiquetas
COLOR_DOODLE = (235, 235, 240)      # Ilustración de fondo

NOMBRES_EJEMPLO = ["Erick Guardo", "Carlos Rodríguez", "María Gómez", "Andrés Martínez", "Valentina López", "Juan David García"]
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

def dibujar_qr_nequi(draw, x, y, size=180):
    pad = 12
    draw.rounded_rectangle([(x - pad, y - pad), (x + size + pad, y + size + pad)], radius=8, fill=COLOR_MINT_QR)
    draw.rounded_rectangle([(x, y), (x + size, y + size)], radius=4, fill=(255, 255, 255))
    
    grid_n = 21
    cell_size = size / grid_n
    np.random.seed(x + y)
    for r in range(grid_n):
        for c in range(grid_n):
            es_esq = (r < 7 and c < 7) or (r < 7 and c >= grid_n - 7) or (r >= grid_n - 7 and c < 7)
            es_cntr = (7 <= r <= 13 and 7 <= c <= 13)
            if es_esq:
                if (r in [0, 6] and 0 <= c <= 6) or (c in [0, 6] and 0 <= r <= 6) or (2 <= r <= 4 and 2 <= c <= 4):
                    draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
                elif (r in [0, 6] and grid_n - 7 <= c < grid_n) or (c in [grid_n - 7, grid_n - 1] and 0 <= r <= 6) or (2 <= r <= 4 and grid_n - 5 <= c <= grid_n - 3):
                    draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
                elif (r in [grid_n - 7, grid_n - 1] and 0 <= c <= 6) or (c in [0, 6] and grid_n - 7 <= r < grid_n) or (grid_n - 5 <= r <= grid_n - 3 and 2 <= c <= 4):
                    draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
            elif not es_cntr and np.random.rand() > 0.45:
                draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
                
    c_x, c_y = x + size/2, y + size/2
    draw.rounded_rectangle([(c_x - 22, c_y - 22), (c_x + 22, c_y + 22)], radius=6, fill=(255, 255, 255))
    draw.text((c_x - 10, c_y - 10), "·N", fill=COLOR_PURPLE)

def crear_comprobante_nequi_actual(datos, ancho=480, alto=880):
    img = Image.new("RGB", (ancho, alto), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    # Líneas perforadas de tiquete
    for x_p in range(15, ancho - 15, 8):
        draw.rectangle([(x_p, 25), (x_p + 4, 27)], fill=(200, 200, 210))
        draw.rectangle([(x_p, alto - 25), (x_p + 4, alto - 23)], fill=(200, 200, 210))
        
    # Fondo ilustrado
    for i in range(150, alto - 50, 45):
        draw.line([(30, i), (ancho - 30, i)], fill=COLOR_DOODLE, width=1)
        
    # 1. QR Superior
    qr_x = (ancho - 190) // 2
    dibujar_qr_nequi(draw, qr_x, 65, size=190)
    
    # 2. Aviso de verificación
    info_y = 310
    draw.ellipse([(qr_x - 15, info_y - 2), (qr_x + 10, info_y + 23)], outline=COLOR_TEXT_DARK, width=2)
    draw.text((qr_x - 4, info_y + 2), "i", fill=COLOR_TEXT_DARK)
    draw.text((qr_x + 18, info_y - 4), "¡Escanea este QR con Nequi para", fill=COLOR_TEXT_DARK)
    draw.text((qr_x + 18, info_y + 14), "verificar tu envío al instante!", fill=COLOR_TEXT_DARK)
    
    # 3. Campos de información
    y_cur = 390
    margen = 55
    
    draw.text((margen, y_cur), "Para", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["nombre"], fill=COLOR_TEXT_DARK)
    
    y_cur += 70
    draw.text((margen, y_cur), "¿Cuánto?", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["monto"], fill=COLOR_TEXT_DARK)
    
    y_cur += 75
    draw.text((margen, y_cur), "Número Nequi", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["telefono"], fill=COLOR_TEXT_DARK)
    
    y_cur += 70
    draw.text((margen, y_cur), "Fecha", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["fecha"], fill=COLOR_TEXT_DARK)
    
    y_cur += 70
    draw.text((margen, y_cur), "Referencia", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["referencia"], fill=COLOR_TEXT_DARK)
    
    return img

def simular_datos_nequi():
    nom = random.choice(NOMBRES_EJEMPLO)
    tel = f"300 {random.randint(100, 999)} {random.randint(1000, 9999)}"
    val = random.choice([20000, 50000, 100000, 150000, 200000, 350000, 500000, 1000000])
    monto_str = f"$ {val:,.2f}".replace(",", "@").replace(".", ",").replace("@", ".")
    f_base = datetime.now() - timedelta(days=random.randint(0, 30), minutes=random.randint(1, 1440))
    hora_12 = f_base.strftime("%I:%M")
    ampm = "a. m." if f_base.hour < 12 else "p. m."
    fecha = f"{f_base.day:02d} de {MESES[f_base.month - 1]} de {f_base.year} a las {hora_12} {ampm}"
    ref = f"M{random.randint(10000000, 99999999)}"
    return {"nombre": nom, "telefono": tel, "monto": monto_str, "monto_num": val, "fecha": fecha, "referencia": ref}

def generar_muestra_moderna(es_fraude=False):
    d = simular_datos_nequi()
    base = crear_comprobante_nequi_actual(d)
    
    if not es_fraude:
        buf = BytesIO()
        base.save(buf, format="JPEG", quality=92)
        buf.seek(0)
        return Image.open(buf)
    else:
        # Simulación de fraude por parche y doble compresión sobre el monto
        buf = BytesIO()
        base.save(buf, format="JPEG", quality=90)
        buf.seek(0)
        img_edit = Image.open(buf).convert("RGB")
        draw = ImageDraw.Draw(img_edit)
        
        # Parche de edición sobre ¿Cuánto?
        draw.rectangle([(50, 480), (380, 525)], fill=(248, 248, 250))
        monto_falso = f"$ {d['monto_num']*5:,.2f}".replace(",", "@").replace(".", ",").replace("@", ".")
        draw.text((54, 484), monto_falso, fill=(10, 10, 15))
        
        buf2 = BytesIO()
        img_edit.save(buf2, format="JPEG", quality=75)
        buf2.seek(0)
        return Image.open(buf2)

# Generar Dataset de Entrenamiento, Validación y Prueba con el NUEVO formato
for split, n_total in [("train", 400), ("val", 80), ("test", 80)]:
    for cls, es_f in [("legitimo", False), ("fraude", True)]:
        folder = f"dataset_nequi_nuevo/{split}/{cls}"
        os.makedirs(folder, exist_ok=True)
        for i in range(n_total // 2):
            img = generar_muestra_moderna(es_fraude=es_f)
            img.save(f"{folder}/{cls}_{i+1:04d}.jpg", quality=90)

print("✓ Dataset con el NUEVO comprobante oficial de Nequi (QR/Ticket) generado exitosamente.")

--- 
### 🔬 3. Análisis Forense Digital: Error Level Analysis (ELA)
Evaluamos la respuesta del algoritmo ELA ante la nueva plantilla de Nequi con fondo claro.

In [ ]:
def calcular_ela(img_pil, calidad=90, escala=15):
    buf = BytesIO()
    img_pil.save(buf, format="JPEG", quality=calidad)
    buf.seek(0)
    recomprimida = Image.open(buf)
    
    dif = ImageChops.difference(img_pil.convert("RGB"), recomprimida)
    max_dif = max([ex[1] for ex in dif.getextrema()]) or 1
    factor = escala * (255.0 / max_dif)
    return ImageEnhance.Brightness(dif).enhance(factor)

# Visualización Lado a Lado
img_leg = generar_muestra_moderna(es_fraude=False)
ela_leg = calcular_ela(img_leg)

img_frd = generar_muestra_moderna(es_fraude=True)
ela_frd = calcular_ela(img_frd)

fig, axs = plt.subplots(2, 2, figsize=(11, 9))
axs[0, 0].imshow(img_leg); axs[0, 0].set_title("Nuevo Comprobante Nequi Legítimo"); axs[0, 0].axis("off")
axs[0, 1].imshow(ela_leg); axs[0, 1].set_title("ELA: Ruido Uniforme (Sin Manipulación)"); axs[0, 1].axis("off")
axs[1, 0].imshow(img_frd); axs[1, 0].set_title("Comprobante con Monto Falsificado"); axs[1, 0].axis("off")
axs[1, 1].imshow(ela_frd); axs[1, 1].set_title("ELA: Anomalía Forense Detectada en Monto", color="red", fontweight="bold"); axs[1, 1].axis("off")
plt.tight_layout()
plt.show()

--- 
### 🧠 4. Arquitectura del Modelo de IA (MobileNetV3 en PyTorch)

In [ ]:
class NequiNuevoDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.samples = []
        self.transform = transform
        for f in glob.glob(f"{root_dir}/{split}/legitimo/*.jpg"):
            self.samples.append((f, 0.0))
        for f in glob.glob(f"{root_dir}/{split}/fraude/*.jpg"):
            self.samples.append((f, 1.0))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img_ela = calcular_ela(img)
        if self.transform:
            img_ela = self.transform(img_ela)
        return img_ela, torch.tensor(label, dtype=torch.float32)

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(NequiNuevoDataset("dataset_nequi_nuevo", "train", transform_train), batch_size=16, shuffle=True)
val_loader = DataLoader(NequiNuevoDataset("dataset_nequi_nuevo", "val", transform_val), batch_size=16, shuffle=False)
test_loader = DataLoader(NequiNuevoDataset("dataset_nequi_nuevo", "test", transform_val), batch_size=16, shuffle=False)

# Compilar Red Neuronal
weights = models.MobileNet_V3_Small_Weights.DEFAULT
modelo_ia = models.mobilenet_v3_small(weights=weights)
in_feats = modelo_ia.classifier[0].in_features

modelo_ia.classifier = nn.Sequential(
    nn.Linear(in_feats, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.4),
    nn.Linear(256, 64),
    nn.ReLU(inplace=True),
    nn.Linear(64, 1)
)
modelo_ia = modelo_ia.to(device)
print("✓ Modelo convolucional adaptado para el nuevo formato Nequi compilado.")

--- 
### 🚀 5. Entrenamiento del Modelo

In [ ]:
criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.AdamW(modelo_ia.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode="min", patience=2, factor=0.5)

epochs = 10
historial = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
mejor_val_loss = float("inf")

for epoch in range(epochs):
    modelo_ia.train()
    t_loss, t_corr, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizador.zero_grad()
        outs = modelo_ia(imgs)
        loss = criterio(outs, labels)
        loss.backward()
        optimizador.step()
        
        t_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(outs) >= 0.5).float()
        t_corr += (preds == labels).sum().item()
        total += labels.size(0)
        
    trn_loss = t_loss / total
    trn_acc = t_corr / total
    
    # Validación
    modelo_ia.eval()
    v_loss, v_corr, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            outs = modelo_ia(imgs)
            loss = criterio(outs, labels)
            v_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(outs) >= 0.5).float()
            v_corr += (preds == labels).sum().item()
            v_total += labels.size(0)
            
    val_loss = v_loss / v_total
    val_acc = v_corr / v_total
    scheduler.step(val_loss)
    
    historial["train_loss"].append(trn_loss); historial["val_loss"].append(val_loss)
    historial["train_acc"].append(trn_acc); historial["val_acc"].append(val_acc)
    
    if val_loss < mejor_val_loss:
        mejor_val_loss = val_loss
        torch.save(modelo_ia.state_dict(), "mejor_modelo_nequi_nuevo.pth")
        
    print(f"Época [{epoch+1:02d}/{epochs:02d}] - Train Loss: {trn_loss:.4f} (Acc: {trn_acc*100:.1f}%) | Val Loss: {val_loss:.4f} (Acc: {val_acc*100:.1f}%)")

print("✓ Modelo entrenado y validado exitosamente.")

--- 
### 📊 6. Evaluación Rigurosa (Matriz de Confusión y Curva ROC-AUC)

In [ ]:
modelo_ia.load_state_dict(torch.load("mejor_modelo_nequi_nuevo.pth"))
modelo_ia.eval()

y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outs = modelo_ia(imgs)
        probs = torch.sigmoid(outs).squeeze(1).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        y_true.extend(labels.numpy())
        y_pred.extend(preds)
        y_probs.extend(probs)

y_true, y_pred, y_probs = np.array(y_true), np.array(y_pred), np.array(y_probs)

print("\n" + "="*50)
print("📈 REPORTE DE CLASIFICACIÓN (MÉTRICAS)")
print("="*50)
print(classification_report(y_true, y_pred, target_names=["Legítimo (0)", "Fraude (1)"]))

# Gráficas
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))

axs[0].plot(historial["train_loss"], label="Train Loss", color="#84E4BD", lw=2)
axs[0].plot(historial["val_loss"], label="Val Loss", color="#200422", lw=2)
axs[0].set_title("Curvas de Pérdida (Loss)")
axs[0].set_xlabel("Época"); axs[0].legend(); axs[0].grid(True, alpha=0.3)

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", cbar=False, ax=axs[1],
            xticklabels=["Legítimo", "Fraude"], yticklabels=["Legítimo", "Fraude"])
axs[1].set_title("Matriz de Confusión")
axs[1].set_xlabel("Predicción"); axs[1].set_ylabel("Etiqueta Real")

fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)
axs[2].plot(fpr, tpr, color="#200422", lw=2, label=f"ROC (AUC = {roc_auc:.3f})")
axs[2].plot([0, 1], [0, 1], color="gray", linestyle="--")
axs[2].set_title("Curva ROC")
axs[2].set_xlabel("FPR"); axs[2].set_ylabel("TPR (Sensibilidad)"); axs[2].legend()
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

--- 
### 📲 7. Módulo de Prueba Interactiva: Sube tu Comprobante de Nequi
Haz clic en **Ejecutar** para subir cualquier captura de comprobante Nequi (como la que descargues de tu app) y verificar su autenticidad.

In [ ]:
from google.colab import files

print("📤 Haz clic en el botón para seleccionar tu comprobante de Nequi:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n🔍 Analizando imagen: {filename}...")
    img_subida = Image.open(filename).convert("RGB")
    img_ela = calcular_ela(img_subida)
    
    t_input = transform_val(img_ela).unsqueeze(0).to(device)
    with torch.no_grad():
        prob_fraude = torch.sigmoid(modelo_ia(t_input)).item()
        
    es_f = prob_fraude >= 0.5
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_subida)
    plt.title("Comprobante Subido"); plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(img_ela)
    color_t = "red" if es_f else "green"
    veredicto = f"ALERTA: POSIBLE FRAUDE ({prob_fraude*100:.1f}%)" if es_f else f"AUTÉNTICO ({(1-prob_fraude)*100:.1f}%)"
    plt.title(f"Diagnóstico: {veredicto}", color=color_t, fontweight="bold"); plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    print("="*55)
    if es_f:
        print(f"🚨 RESULTADO: [FRAUDE / MANIPULACIÓN DETECTADA]")
        print(f"   Probabilidad de Fraude: {prob_fraude*100:.2f}%")
        print("   Se encontraron discrepancias en la tasa de compresión ELA o plantilla.")
    else:
        print(f"✅ RESULTADO: [COMPROBANTE AUTÉNTICO]")
        print(f"   Probabilidad de Autenticidad: {(1-prob_fraude)*100:.2f}%")
        print("   La imagen presenta textura de compresión homogénea y formato oficial.")
    print("="*55)